# Python 基础语法复习 — 第一阶段（1-2 个月）

这一阶段的目标不是刷高级技巧，而是把“能读懂、能写对、能调试”的基本功补齐。

| 模块 | 重点 |
|---|---|
| 变量与数据类型 | `int` / `float` / `str` / `bool` / `None` |
| 条件与循环 | `if` / `for` / `while` / `break` / `continue` |
| 函数 | 参数、默认值、返回值、作用域 |
| 容器 | `list` / `dict` / `set` / tuple |
| 文件与异常 | `with open(...)`、`try/except` |
| OOP 基础 | 类、对象、属性、方法、`__init__` |

推荐资源：Python.org 官方教程、《Python 编程：从入门到实践》。

---
## 1. 变量、数据类型、条件与循环

### 必须掌握
- Python 变量是“名字绑定到对象”，不是固定类型的盒子
- `==` 比较值，`is` 比较是否同一个对象；判断 `None` 用 `is None`
- `for` 常用于遍历可迭代对象，`while` 常用于条件未知的循环
- 写条件时优先清晰，不要把多个复杂判断塞进一行

In [ ]:
name = "orders"
row_count = 1250
success_rate = 0.98
is_valid = row_count > 0 and success_rate >= 0.95

if is_valid:
    print(f"{name}: ready, rows={row_count}")
else:
    print(f"{name}: needs review")

total = 0
for value in [10, 20, 30]:
    total += value
print("total =", total)

attempt = 0
while attempt < 3:
    attempt += 1
    print("try", attempt)

---
## 2. 函数、参数、返回值

函数是工程化的第一步：把重复逻辑命名、拆小、隔离副作用。

### 复习要点
- 一个函数最好只做一件事
- 参数名要表达业务含义，不要只写 `x` / `data`
- 能返回值就返回值，少依赖全局变量
- 默认参数不要用可变对象，比如 `items=[]`

In [ ]:
def calculate_discounted_amount(amount: float, discount: float = 0.0) -> float:
    if amount < 0:
        raise ValueError("amount cannot be negative")
    if not 0 <= discount <= 1:
        raise ValueError("discount must be between 0 and 1")
    return round(amount * (1 - discount), 2)

print(calculate_discounted_amount(100, 0.2))

# Bad: default list is shared across calls
# def append_item(item, items=[]): ...

# Good: use None as default sentinel
def append_item(item: str, items: list[str] | None = None) -> list[str]:
    if items is None:
        items = []
    items.append(item)
    return items

print(append_item("csv"))
print(append_item("json"))

---
## 3. List / Dict / Set 操作

| 类型 | 常用场景 | 关键操作 |
|---|---|---|
| `list` | 有顺序、允许重复 | append, extend, slice, list comprehension |
| `dict` | key-value 查找 | get, items, keys, values, update |
| `set` | 去重、集合关系 | add, union, intersection, difference |

数据工程里，`dict` 常用来做映射表，`set` 常用来去重和检查是否存在。

In [ ]:
orders = [
    {"order_id": 1, "status": "paid", "amount": 100},
    {"order_id": 2, "status": "failed", "amount": 0},
    {"order_id": 3, "status": "paid", "amount": 250},
]

paid_amounts = [o["amount"] for o in orders if o["status"] == "paid"]
print("paid_amounts:", paid_amounts)

status_label = {"paid": "已支付", "failed": "失败"}
for order in orders:
    label = status_label.get(order["status"], "未知")
    print(order["order_id"], label)

raw_user_ids = [101, 102, 101, 103, 102]
unique_user_ids = set(raw_user_ids)
print("unique users:", unique_user_ids)

---
## 4. 文件读写与异常处理

### 必须养成的习惯
- 文件读写用 `with`，让 Python 自动关闭文件句柄
- 明确指定 `encoding="utf-8"`
- 异常不要裸写 `except:`，要捕获具体异常
- 不要吞掉异常后什么都不做，至少记录或重新抛出

In [ ]:
from pathlib import Path
import json
import tempfile

records = [
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"},
]

path = Path(tempfile.gettempdir()) / "sample_records.json"

with path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

try:
    with path.open("r", encoding="utf-8") as f:
        loaded = json.load(f)
except FileNotFoundError:
    loaded = []

print(loaded)

---
## 5. 类与对象（OOP 基础）

先掌握“把数据和行为放在一起”，不用急着学复杂继承。

### 复习要点
- `class` 定义类型，实例是对象
- `__init__` 初始化对象状态
- 方法第一个参数通常是 `self`
- 数据对象优先考虑 `dataclass`，比手写样板代码更清晰

In [ ]:
from dataclasses import dataclass

@dataclass
class Order:
    order_id: int
    amount: float
    status: str

    def is_successful(self) -> bool:
        return self.status == "paid" and self.amount > 0

order = Order(order_id=1, amount=120.5, status="paid")
print(order)
print(order.is_successful())

---
## 阶段验收

完成第一阶段后，你应该能独立做到：

1. 读懂普通 Python 脚本，不被语法卡住
2. 写函数处理 list/dict 数据，并返回清晰结果
3. 读写 JSON/CSV 文本文件，能处理常见异常
4. 用 class 或 dataclass 表达一个简单业务对象
5. 遇到错误能看 traceback，定位到具体文件和行号

练习项目：读取一个订单 JSON 文件，过滤成功订单，计算总金额，然后写出汇总 JSON。